# Lezione 14 — Pratica: Web Scraping con BeautifulSoup

Questi esercizi rafforzano quanto visto a lezione: navigare l'albero HTML con BeautifulSoup, filtrare con `find()`/`find_all()`, estrarre una tabella a mano cella per cella, ed estrarre la stessa tabella con `pd.read_html()`.

Tutto l'HTML di questi esercizi è definito come stringa Python — non serve connessione a internet.

Per ogni esercizio:
1. Leggi l'enunciato e i suggerimenti.
2. Completa il codice dove dice `# TODO`.
3. Esegui la cella e confronta il risultato con quanto atteso nell'enunciato.
4. Se ti blocchi, apri la soluzione nascosta sotto — ma prova a risolverlo prima da solo.

In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
import io

## Esercizio 1 — Navigare il profilo di un dipendente (Tag, attributi, siblings)

Questo è l'HTML di una scheda dipendente fittizia:

```python
html_empleado = """
<div>
<h3 id="nombre_completo">Ana Torres</h3>
<p class="puesto">Ruolo: Analista Dati</p>
<p class="antiguedad">Anzianità: 3 anni</p>
</div>
"""
```

Completa il codice affinché:
1. Crei l'oggetto `BeautifulSoup` a partire da `html_empleado`, usando il parser `'html.parser'`.
2. Ottenga il tag `<h3>` ed estragga il suo **attributo `id`** con le parentesi quadre.
3. Ottenga il **testo** dell'`<h3>` (il nome "Ana Torres") con `.string` o `.get_text()`.
4. A partire dall'`<h3>`, usi `.find_next_sibling()` per arrivare al `<p>` del "ruolo", e da lì di nuovo `.find_next_sibling()` per arrivare al `<p>` dell'"anzianità".

**Suggerimenti:**
- Il costruttore è `BeautifulSoup(html, 'html.parser')`.
- Accedere a un tag per nome (`soup.h3`) ti dà il primo che compare nel documento.
- L'attributo si accede come un dizionario: `tag['id']`.
- Dato che l'HTML originale ha ritorni a capo tra i tag, `.next_sibling` "puro" può restituirti quel ritorno a capo invece del tag che cerchi. `.find_next_sibling()` salta direttamente al **tag successivo**, ignorando gli spazi bianchi isolati.

In [ ]:
html_empleado = """
<div>
<h3 id="nombre_completo">Ana Torres</h3>
<p class="puesto">Ruolo: Analista Dati</p>
<p class="antiguedad">Anzianità: 3 anni</p>
</div>
"""

# TODO: crea l'oggetto BeautifulSoup usando 'html.parser'
soup_empleado = None

# TODO: ottieni il tag <h3> (accesso diretto per nome di tag)
tag_h3 = None
print("tag h3:", tag_h3)

# TODO: ottieni il valore dell'attributo id di tag_h3, usando le parentesi quadre
id_h3 = None
print("id:", id_h3)

# TODO: ottieni il testo contenuto in tag_h3 (il nome del dipendente)
nombre_empleado = None
print("nome:", nombre_empleado)

# TODO: usa find_next_sibling() su tag_h3 per arrivare al <p> del "ruolo"
tag_puesto = None
print("ruolo:", tag_puesto)

# TODO: da tag_puesto, usa find_next_sibling() per arrivare al <p> dell'"anzianità"
tag_antiguedad = None
print("anzianità:", tag_antiguedad)

<details>
<summary>💡 Vedi soluzione</summary>

```python
html_empleado = """
<div>
<h3 id="nombre_completo">Ana Torres</h3>
<p class="puesto">Ruolo: Analista Dati</p>
<p class="antiguedad">Anzianità: 3 anni</p>
</div>
"""

# Creiamo l'oggetto BeautifulSoup con il parser incluso nella libreria standard
soup_empleado = BeautifulSoup(html_empleado, 'html.parser')

# soup.h3 restituisce il primo (e unico) <h3> del documento
tag_h3 = soup_empleado.h3
print("tag h3:", tag_h3)

# Un tag si comporta come un dizionario per i suoi attributi
id_h3 = tag_h3['id']
print("id:", id_h3)

# Il testo interno del tag è un NavigableString
nombre_empleado = tag_h3.get_text()
print("nome:", nombre_empleado)

# find_next_sibling() avanza al PROSSIMO TAG allo stesso livello dell'albero, saltando
# i ritorni a capo/spazi bianchi che .next_sibling "puro" restituirebbe invece
tag_puesto = tag_h3.find_next_sibling()
print("ruolo:", tag_puesto)

tag_antiguedad = tag_puesto.find_next_sibling()
print("anzianità:", tag_antiguedad)
```
</details>

## Esercizio 2 — Filtrare recensioni di film con `find_all()`

Questo è l'HTML di una lista di recensioni di film:

```python
html_resenas = """
<div>
  <p class="resena" data-estrellas="5">Un capolavoro del cinema.</p>
  <p class="resena" data-estrellas="2">Non ha soddisfatto le aspettative.</p>
  <p class="resena" data-estrellas="5">La rivedrei mille volte.</p>
  <p class="resena" data-estrellas="3">Divertente ma dimenticabile.</p>
  <a href="https://ejemplo.com/pelicula/1">Più info</a>
  <a>Commenti chiusi</a>
</div>
"""
```

Completa il codice affinché:
1. Crei l'oggetto `BeautifulSoup` a partire da `html_resenas`.
2. Usi `find_all()` per ottenere **tutti** i `<p>` con `class_="resena"`.
3. Usi `find_all()` con l'attributo `data-estrellas="5"` per tenere solo le recensioni a 5 stelle (suggerimento: gli attributi con trattino si passano come argomento con nome usando `**{"data-estrellas": "5"}`, oppure direttamente come dizionario nel parametro `attrs`).
4. Usi `find_all('a', href=True)` per trovare i link che hanno `href`, e `find_all('a', href=False)` per quelli che non ce l'hanno.

**Suggerimenti:**
- `class` è parola riservata in Python, per questo si filtra con `class_` (trattino basso finale).
- Per attributi con trattino nel nome (come `data-estrellas`), non puoi scrivere `data-estrellas="5"` direttamente come argomento Python perché il trattino non è valido in un nome di parametro; usa `attrs={"data-estrellas": "5"}`.
- `find_all('a', href=True)` filtra per "ha l'attributo", indipendentemente dal valore esatto.

In [ ]:
html_resenas = """
<div>
  <p class="resena" data-estrellas="5">Un capolavoro del cinema.</p>
  <p class="resena" data-estrellas="2">Non ha soddisfatto le aspettative.</p>
  <p class="resena" data-estrellas="5">La rivedrei mille volte.</p>
  <p class="resena" data-estrellas="3">Divertente ma dimenticabile.</p>
  <a href="https://ejemplo.com/pelicula/1">Più info</a>
  <a>Commenti chiusi</a>
</div>
"""

# TODO: crea l'oggetto BeautifulSoup a partire da html_resenas
soup_resenas = None

# TODO: trova tutti i <p> con class_="resena"
todas_las_resenas = None
print("recensioni trovate:", todas_las_resenas)

# TODO: trova solo le recensioni con attributo data-estrellas uguale a "5"
# usa il parametro attrs con un dizionario, es: attrs={"data-estrellas": "5"}
resenas_5_estrellas = None
print("recensioni a 5 stelle:", resenas_5_estrellas)

# TODO: trova gli <a> che hanno href
links_con_href = None
print("link con href:", links_con_href)

# TODO: trova gli <a> che NON hanno href
links_sin_href = None
print("link senza href:", links_sin_href)

<details>
<summary>💡 Vedi soluzione</summary>

```python
html_resenas = """
<div>
  <p class="resena" data-estrellas="5">Un capolavoro del cinema.</p>
  <p class="resena" data-estrellas="2">Non ha soddisfatto le aspettative.</p>
  <p class="resena" data-estrellas="5">La rivedrei mille volte.</p>
  <p class="resena" data-estrellas="3">Divertente ma dimenticabile.</p>
  <a href="https://ejemplo.com/pelicula/1">Più info</a>
  <a>Commenti chiusi</a>
</div>
"""

soup_resenas = BeautifulSoup(html_resenas, 'html.parser')

# class_ (con trattino basso) perché "class" è parola riservata in Python
todas_las_resenas = soup_resenas.find_all('p', class_="resena")
print("quantità di recensioni:", len(todas_las_resenas))

# Gli attributi con trattino nel nome si passano dentro un dizionario in attrs
resenas_5_estrellas = soup_resenas.find_all('p', attrs={"data-estrellas": "5"})
print("recensioni a 5 stelle:")
for r in resenas_5_estrellas:
    print("-", r.get_text())

# href=True filtra "ha l'attributo href", indipendentemente dal valore
links_con_href = soup_resenas.find_all('a', href=True)
print("link con href:", links_con_href)

# href=False filtra quelli che NON hanno l'attributo
links_sin_href = soup_resenas.find_all('a', href=False)
print("link senza href:", links_sin_href)
```
</details>

## Esercizio 3 — Una tabella di dipendenti: a mano vs. `pd.read_html()`

Questo è l'HTML di una tabella di dipendenti di un'azienda fittizia:

```python
html_tabla_empleados = """
<table id="tabla_staff">
  <tr>
    <th>Nome</th>
    <th>Reparto</th>
    <th>Stipendio</th>
  </tr>
  <tr>
    <td>Ana Torres</td>
    <td>Dati</td>
    <td>52000</td>
  </tr>
  <tr>
    <td>Luis Peña</td>
    <td>Vendite</td>
    <td>41000</td>
  </tr>
  <tr>
    <td>Marta Ruiz</td>
    <td>Dati</td>
    <td>58000</td>
  </tr>
</table>
"""
```

Completa il codice affinché:
1. **A mano con BeautifulSoup**: crei il `soup`, trovi la tabella con `find('table')`, percorra le sue righe con `find_all('tr')` (saltando l'intestazione) e, per ogni riga, estragga nome, reparto e stipendio in una lista di tuple chiamata `empleados`.
2. Converta quella lista di tuple `empleados` in un DataFrame di pandas con colonne `["Nome", "Reparto", "Stipendio"]`, chiamato `df_manual`.
3. **Con pandas diretto**: usi `pd.read_html(io.StringIO(html_tabla_empleados))` per ottenere la stessa tabella in un DataFrame chiamato `df_pandas`, senza percorrere nulla a mano.
4. Confronti entrambi i DataFrame stampando le loro colonne e il numero di righe — dovrebbero coincidere nel contenuto, anche se `df_pandas` porterà le intestazioni già usate automaticamente come nomi di colonna.

**Suggerimenti:**
- Lo schema di sempre: `tabla.find_all('tr')` per le righe, e dentro ogni riga `fila.find_all('td')` per le celle.
- La prima riga è l'intestazione ("Nome", "Reparto", "Stipendio"), non un dipendente — va saltata durante il percorso (per esempio con `filas[1:]`).
- `pd.DataFrame(lista_de_tuplas, columns=[...])` costruisce un DataFrame a partire da una lista di tuple.
- `pd.read_html()` moderno raccomanda di avvolgere la stringa HTML con `io.StringIO(...)`.

In [ ]:
html_tabla_empleados = """
<table id="tabla_staff">
  <tr>
    <th>Nome</th>
    <th>Reparto</th>
    <th>Stipendio</th>
  </tr>
  <tr>
    <td>Ana Torres</td>
    <td>Dati</td>
    <td>52000</td>
  </tr>
  <tr>
    <td>Luis Peña</td>
    <td>Vendite</td>
    <td>41000</td>
  </tr>
  <tr>
    <td>Marta Ruiz</td>
    <td>Dati</td>
    <td>58000</td>
  </tr>
</table>
"""

# TODO: crea il soup con 'html.parser'
soup_staff = None

# TODO: trova la tabella con find('table')
tabla_staff = None

# TODO: ottieni tutte le righe con find_all('tr')
filas_staff = None

# TODO: percorri le righe SALTANDO l'intestazione (filas_staff[1:]) e costruisci
# una lista di tuple (nome, reparto, stipendio) chiamata 'empleados'
empleados = []
# il tuo ciclo for qui

print("dipendenti estratti a mano:", empleados)

# TODO: converti 'empleados' in un DataFrame chiamato df_manual
# con columns=["Nome", "Reparto", "Stipendio"]
df_manual = None
print(df_manual)

In [ ]:
# TODO: usa pd.read_html(io.StringIO(html_tabla_empleados)) per ottenere
# la lista delle tabelle, e salva la prima in df_pandas
tablas_staff = None
df_pandas = None
print(df_pandas)

# TODO: stampa le colonne e il numero di righe di entrambi i DataFrame per confrontarli
print("colonne df_manual:", None)
print("colonne df_pandas:", None)
print("righe df_manual:", None)
print("righe df_pandas:", None)

<details>
<summary>💡 Vedi soluzione</summary>

```python
html_tabla_empleados = """
<table id="tabla_staff">
  <tr>
    <th>Nome</th>
    <th>Reparto</th>
    <th>Stipendio</th>
  </tr>
  <tr>
    <td>Ana Torres</td>
    <td>Dati</td>
    <td>52000</td>
  </tr>
  <tr>
    <td>Luis Peña</td>
    <td>Vendite</td>
    <td>41000</td>
  </tr>
  <tr>
    <td>Marta Ruiz</td>
    <td>Dati</td>
    <td>58000</td>
  </tr>
</table>
"""

soup_staff = BeautifulSoup(html_tabla_empleados, 'html.parser')

# find() ci dà l'unica tabella del documento
tabla_staff = soup_staff.find('table')

# find_all('tr') porta tutte le righe, inclusa l'intestazione
filas_staff = tabla_staff.find_all('tr')

empleados = []
for fila in filas_staff[1:]:   # [1:] salta la riga di intestazione
    cols = fila.find_all('td')
    nombre = cols[0].get_text()
    departamento = cols[1].get_text()
    salario = cols[2].get_text()
    empleados.append((nombre, departamento, salario))

print("dipendenti estratti a mano:", empleados)

df_manual = pd.DataFrame(empleados, columns=["Nome", "Reparto", "Stipendio"])
print(df_manual)
```

```python
import io

# pd.read_html fa tutto il lavoro di rilevare e costruire il DataFrame per noi
tablas_staff = pd.read_html(io.StringIO(html_tabla_empleados))
df_pandas = tablas_staff[0]
print(df_pandas)

# Confrontiamo entrambi i risultati
print("colonne df_manual:", list(df_manual.columns))
print("colonne df_pandas:", list(df_pandas.columns))
print("righe df_manual:", len(df_manual))
print("righe df_pandas:", len(df_pandas))
# Entrambi i DataFrame portano gli stessi 3 dipendenti; pd.read_html inoltre ha rilevato
# automaticamente che la prima riga della tabella era l'intestazione, e l'ha usata
# come nomi di colonna senza che dovessimo indicarglielo.
```
</details>